In [4]:
# =========================================
# Common setup (TensorFlow / Keras)
# =========================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---- Paths & data ----
DATA_DIR =  "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Brain Cancer"  # directory with subfolders per class
IMG_SIZE = (256, 256)                    # you can change to (224,224) if you like
BATCH_SIZE = 8
SEED = 1337

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED
)

# Split val/test
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# =========================================
# Classifier head (shared)
# =========================================
def classifier_head(x, num_classes, dropout=0.2):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize(x)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (≈ GoogLeNet successor, prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    # InceptionV3 was designed for 299x299, but works with other >=75 sizes in Keras.
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    # EfficientNet expects its own scaling if using `keras.applications.efficientnet.preprocess_input`,
    # but simple Rescaling(1/255) works fine for transfer learning in practice.
    x = normalize(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train
# =========================================
MODEL_NAME = "efficientnet"  # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"         # "imagenet" for transfer learning, or None to train from scratch
FINE_TUNE_BASE = False       # set True to unfreeze backbone (after a few epochs usually)

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)


# Evaluate
test_loss, test_acc = model.evaluate(val_ds, verbose=0)
print(f"{model.name}  |  Val acc: {test_acc:.4f}  |  Val loss: {test_loss:.4f}")


Found 15000 files belonging to 3 classes.
Using 12750 files for training.
Found 15000 files belonging to 3 classes.
Using 2250 files for validation.
Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Epoch 1/20
1594/1594 [==============================] - 30s 15ms/step - loss: 1.1291 - accuracy: 0.3354 - val_loss: 0.7776 - val_accuracy: 1.0000
Epoch 2/20
1594/1594 [==============================] - 23s 15ms/step - loss: 1.1268 - accuracy: 0.3330 - val_loss: 0.7250 - val_accuracy: 1.0000
Epoch 3/20
1594/1594 [==============================] - 23s 15ms/step - loss: 1.1249 - accuracy: 0.3336 - val_loss: 0.7070 - val_accuracy: 1.0000
Epoch 4/20
1594/1594 [==============================] - 23s 15ms/step - loss: 1.1229 - accuracy: 0.3365 - val_loss: 1.0509 - val_accuracy: 0.9173
Epoch 5/20
1594/1594 [==============================] - 23s 15ms/step - loss: 1.1243 - accuracy: 0.3363 - val_loss: 0.7305 - val_accuracy: 1.0000
efficientnetb0  |  Val acc: 1.0000  |  Val loss: 0.7776
